In [1]:
import cv2
import numpy as np
import torch as t
import torchvision
import torchaudio
import albumentations as A
from ultralytics import YOLO

In [5]:
pts_real_3d = np.array([
    [0.0, 0.0, 0.0],
    [9.0, 0.0, 0.0],
    [9.0, 18.0, 0.0],
    [0.0, 18.0, 0.0]],dtype=np.float32)

pts_video_2d = np.array([
    [73, 1031],
    [1835, 1027],
    [1504, 606],
    [405, 606]],dtype=np.float32)

K = np.array([
    [1300.0, 0.0,    960.0],
    [0.0,    1300.0, 540.0],
    [0.0,    0.0,    1.0]],dtype=np.float32)

dist_coeffs = np.zeros((4, 1))

def calibrate_camera(pts_3d, pts_2d, camera_matrix, dist):
    success, rvec, tvec = cv2.solvePnP(pts_3d, pts_2d, camera_matrix, dist, flags=cv2.SOLVEPNP_ITERATIVE)

    if not success:
        raise ValueError("solvePnP не зміг знайти рішення! Перевір порядок точок.")

    R, _ = cv2.Rodrigues(rvec)

    # Знаходимо фізичну позицію камери в залі (C = -R^T * tvec)
    R_inv = np.linalg.inv(R)
    camera_position = -np.dot(R_inv, tvec)

    return R, tvec, camera_position

# Виконуємо калібрування один раз для всього відео
R_matrix, t_vec, camera_pos = calibrate_camera(pts_real_3d, pts_video_2d, K, dist_coeffs)
# d_cam = np.matmul(np.linalg.inv(K), )

print("Матриця обертання R:\n", R_matrix)
print("\nПозиція камери в залі (X, Y, Z) в метрах:\n", camera_pos.ravel())

Матриця обертання R:
 [[ 9.99984849e-01 -5.49723594e-03  2.87047282e-04]
 [-1.04674182e-04 -7.11253619e-02 -9.97467379e-01]
 [ 5.50372986e-03  9.97452236e-01 -7.11248596e-02]]

Позиція камери в залі (X, Y, Z) в метрах:
 [ 4.47646972 -6.29310278  2.8990669 ]


In [ ]:
model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/training_models/models/main_model_april.pt')

